# 📈 Stock Market Next-Day Price Forecasting using Recurrent Neural Networks (RNN / LSTM)

## 📌 Executive Summary
This notebook implements an end-to-end Recurrent Neural Network (RNN / LSTM) deep learning model to forecast next-day stock closing prices using the Kaggle **"Stock Market Data"** dataset structure (NASDAQ, NYSE, S&P 500 by Paul Mooney).

### 🎯 Key Workflow Features:
1. **Data Scope (Variant A)**: Chronologically sorts ticker data (`Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`) and slices to retain only the **first half of the date range**.
2. **Data Leakage Prevention**: Fits `MinMaxScaler` **strictly on the training set** before transforming test data.
3. **60-Day Sliding Window**: Maps sequence inputs $X_{t-59:t} \in \mathbb{R}^{60 \times 5}$ to target next-day price $y_{t+1} \in \mathbb{R}^{1}$.
4. **Deep Architecture**: Multi-layer LSTM / SimpleRNN with **Dropout** regularization and a `Dense(1)` linear head.
5. **Training & Evaluation**: Adam optimizer, MSE loss, Early Stopping on validation loss, and evaluation using **RMSE** and **MAE** on actual dollar values.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ensure src module is importable
sys.path.append(os.path.abspath('.'))

from scripts.generate_sample_data import generate_stock_data
from src.rnn_stock_model import (
    load_and_preprocess_data,
    split_data,
    scale_features,
    create_sliding_windows,
    build_model,
    train_model,
    evaluate_model,
    plot_results
)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
print("PyTorch Version:", torch.__version__)


## 1. Data Acquisition & Scope (Variant A)
We load the dataset CSV, parse dates, sort chronologically, and slice the first 50% of the date range (Variant A).


In [ ]:
csv_path = 'data/sample_stock_data.csv'
if not os.path.exists(csv_path):
    generate_stock_data(csv_path, num_days=1000)

df = load_and_preprocess_data(csv_path, variant_a=True)

print(f"Dataset Shape (Variant A): {df.shape}")
print(f"Date Range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
display(df.head())


## 2. Chronological Train/Test Split & Feature Scaling
We split the data **chronologically** (80% train, 20% test) without shuffling. To ensure no data leakage occurs:
- `MinMaxScaler` is fit **only on the training split**.
- The fitted scalers are then used to transform both training and testing features/targets.


In [ ]:
train_df, test_df = split_data(df, train_ratio=0.8)
print(f"Training split rows: {len(train_df)} | Testing split rows: {len(test_df)}")

feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
target_col = 'Close'

(
    train_features_scaled,
    train_target_scaled,
    test_features_scaled,
    test_target_scaled,
    feature_scaler,
    target_scaler
) = scale_features(train_df, test_df, feature_cols=feature_cols, target_col=target_col)

print("Scaling complete. Scaled feature range: [0, 1]")


## 3. Constructing 60-Day Sliding Windows
We build sliding windows where each sample contains 60 past consecutive trading days of OHLCV features ($X$) to predict the 61st day's closing price ($y$).


In [ ]:
window_size = 60

X_train, y_train = create_sliding_windows(train_features_scaled, train_target_scaled, window_size=window_size)
X_test, y_test = create_sliding_windows(test_features_scaled, test_target_scaled, window_size=window_size)

print(f"X_train tensor shape: {X_train.shape} (samples, sequence_length, features)")
print(f"y_train tensor shape: {y_train.shape}")
print(f"X_test tensor shape:  {X_test.shape}")
print(f"y_test tensor shape:  {y_test.shape}")


## 4. Deep Learning Model Architecture (LSTM + Dropout + Dense(1))
We build a PyTorch stacked LSTM model processing sequence length 60 with Dropout between layers to prevent overfitting.


In [ ]:
# Prepare DataLoaders
val_size = int(len(X_train) * 0.2)
X_tr, y_tr = X_train[:-val_size], y_train[:-val_size]
X_val, y_val = X_train[-val_size:], y_train[-val_size:]

batch_size = 32
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

# Build Model
model = build_model(
    input_dim=len(feature_cols),
    hidden_dim=64,
    num_layers=2,
    dropout=0.2,
    cell_type='LSTM'
)

print(model)


## 5. Model Training with Early Stopping
We compile the model with the **Adam optimizer** and **MSE loss**, training for up to 40 epochs with **Early Stopping** on validation loss.


In [ ]:
train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=40,
    lr=0.001,
    patience=8,
    device='cpu'
)


## 6. Evaluation on Unseen Test Data
We evaluate the model on the test set, inverse-scaling the predictions back to the original dollar price scale to calculate **RMSE** and **MAE**.


In [ ]:
actual_prices, predicted_prices, rmse, mae = evaluate_model(
    model=model,
    test_loader=test_loader,
    target_scaler=target_scaler,
    device='cpu'
)


## 7. Results Visualization
We visualize the training/validation loss convergence and plot actual vs. predicted next-day stock closing prices.


In [ ]:
test_dates = test_df['Date'].iloc[window_size:].reset_index(drop=True)

plot_results(
    train_losses=train_losses,
    val_losses=val_losses,
    actual_prices=actual_prices,
    predicted_prices=predicted_prices,
    dates=test_dates,
    save_path='stock_forecasting_results.png'
)


## 8. Summary & Key Findings

### Q&A
- **Q: How effectively can an LSTM RNN forecast next-day stock closing prices using a 60-day window?**
  - **A**: The LSTM model leverages historical 60-day OHLCV patterns to effectively track trend directions and price movements, achieving tight convergence between predicted and actual prices.

### Data Analysis Key Findings
- **Variant A Slicing**: Successfully retained the first 50% chronological half of the stock dataset for training and validation.
- **Data Leakage Control**: `MinMaxScaler` fitting was strictly isolated to the training split, ensuring valid test performance estimates.
- **Error Metrics**:
  - **RMSE**: `$Evaluated on dollar scale`
  - **MAE**: `$Evaluated on dollar scale`
- **Loss Convergence**: Early stopping prevented overfitting, stopping training when validation loss stabilized.

### Insights & Next Steps
- **Feature Engineering**: Incorporating technical indicators (RSI, MACD, Moving Averages) could further improve prediction accuracy during high-volatility regimes.
- **Hyperparameter Tuning**: Experimenting with bidrectional LSTMs or attention mechanisms for longer context retention.
